In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score

import lightgbm as lgb

RANDOM_STATE = 42

print("Imports successful.")
print("LightGBM:", lgb.__version__)

Imports successful.
LightGBM: 4.7.0


In [2]:
previous = pd.read_csv("../data/raw/previous_application.csv")

print("Shape:", previous.shape)
print("Unique applicants:", previous["SK_ID_CURR"].nunique())

print("\nApplications per applicant:")
print(previous.groupby("SK_ID_CURR").size().describe())

Shape: (1670214, 37)
Unique applicants: 338857

Applications per applicant:
count    338857.000000
mean          4.928964
std           4.220716
min           1.000000
25%           2.000000
50%           4.000000
75%           7.000000
max          77.000000
dtype: float64


In [3]:
print("Columns:", previous.shape[1])
print("\nFirst 5 rows:")
display(previous.head())

print("\nContract status:")
print(previous["NAME_CONTRACT_STATUS"].value_counts())

print("\nContract type:")
print(previous["NAME_CONTRACT_TYPE"].value_counts())

print("\nYield group:")
print(previous["NAME_YIELD_GROUP"].value_counts())

print("\nTop missing-value rates:")
print(
    previous.isna()
    .mean()
    .sort_values(ascending=False)
    .head(10)
)

Columns: 37

First 5 rows:


,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,...,NAME_SELLER_INDUSTRY,CNT_PAYMENT,NAME_YIELD_GROUP,PRODUCT_COMBINATION,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,0.0,17145.0,SATURDAY,15,...,Connectivity,12.0,middle,POS mobile with interest,365243.0,-42.0,300.0,-42.0,-37.0,0.0
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,NaN,607500.0,THURSDAY,11,...,XNA,36.0,low_action,Cash X-Sell: low,365243.0,-134.0,916.0,365243.0,365243.0,1.0
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,NaN,112500.0,TUESDAY,11,...,XNA,12.0,high,Cash X-Sell: high,365243.0,-271.0,59.0,365243.0,365243.0,1.0
3,2819243,176158,Cash loans,47041.335,450000.0,470790.0,NaN,450000.0,MONDAY,7,...,XNA,12.0,middle,Cash X-Sell: middle,365243.0,-482.0,-152.0,-182.0,-177.0,1.0
4,1784265,202054,Cash loans,31924.395,337500.0,404055.0,NaN,337500.0,THURSDAY,9,...,XNA,24.0,high,Cash Street: high,NaN,NaN,NaN,NaN,NaN,NaN



Contract status:
NAME_CONTRACT_STATUS
Approved        1036781
Canceled         316319
Refused          290678
Unused offer      26436
Name: count, dtype: int64

Contract type:
NAME_CONTRACT_TYPE
Cash loans         747553
Consumer loans     729151
Revolving loans    193164
XNA                   346
Name: count, dtype: int64

Yield group:
NAME_YIELD_GROUP
XNA           517215
middle        385532
high          353331
low_normal    322095
low_action     92041
Name: count, dtype: int64

Top missing-value rates:
RATE_INTEREST_PRIVILEGED     0.996437
RATE_INTEREST_PRIMARY        0.996437
AMT_DOWN_PAYMENT             0.536365
RATE_DOWN_PAYMENT            0.536365
NAME_TYPE_SUITE              0.491198
DAYS_TERMINATION             0.402981
DAYS_FIRST_DRAWING           0.402981
DAYS_FIRST_DUE               0.402981
DAYS_LAST_DUE_1ST_VERSION    0.402981
DAYS_LAST_DUE                0.402981
dtype: float64


In [4]:
def aggregate_previous(previous):
    p = previous.copy()

    # Numerical features
    num_agg = p.groupby("SK_ID_CURR").agg({
        "SK_ID_PREV": ["count"],
        "AMT_ANNUITY": ["mean", "max"],
        "AMT_APPLICATION": ["mean", "max"],
        "AMT_CREDIT": ["mean", "max"],
        "AMT_DOWN_PAYMENT": ["mean"],
        "AMT_GOODS_PRICE": ["mean"],
        "DAYS_DECISION": ["mean", "min", "max"],
    })

    num_agg.columns = [
        "PREV_" + "_".join(c).upper()
        for c in num_agg.columns
    ]

    # Contract status counts
    status = pd.crosstab(
        p["SK_ID_CURR"],
        p["NAME_CONTRACT_STATUS"]
    )

    for col in [
        "Approved",
        "Canceled",
        "Refused",
        "Unused offer"
    ]:
        if col not in status.columns:
            status[col] = 0

    status = status[
        ["Approved", "Canceled", "Refused", "Unused offer"]
    ]

    status.columns = [
        "PREV_STATUS_" + col.upper().replace(" ", "_")
        for col in status.columns
    ]

    # Approval / refusal / cancellation rates
    total = status.sum(axis=1)

    status["PREV_APPROVAL_RATE"] = (
        status["PREV_STATUS_APPROVED"] / total
    )

    status["PREV_REFUSAL_RATE"] = (
        status["PREV_STATUS_REFUSED"] / total
    )

    status["PREV_CANCELLATION_RATE"] = (
        status["PREV_STATUS_CANCELED"] / total
    )

    # Combine everything
    result = num_agg.join(status, how="left")

    return result.reset_index()

prev_agg = aggregate_previous(previous)

print("Original rows:", len(previous))
print("Aggregated rows:", len(prev_agg))
print("Unique applicants:", prev_agg["SK_ID_CURR"].nunique())
print("Aggregated columns:", prev_agg.shape[1])

display(prev_agg.head())

Original rows: 1670214
Aggregated rows: 338857
Unique applicants: 338857
Aggregated columns: 20


,SK_ID_CURR,PREV_SK_ID_PREV_COUNT,PREV_AMT_ANNUITY_MEAN,PREV_AMT_ANNUITY_MAX,PREV_AMT_APPLICATION_MEAN,PREV_AMT_APPLICATION_MAX,PREV_AMT_CREDIT_MEAN,PREV_AMT_CREDIT_MAX,PREV_AMT_DOWN_PAYMENT_MEAN,PREV_AMT_GOODS_PRICE_MEAN,PREV_DAYS_DECISION_MEAN,PREV_DAYS_DECISION_MIN,PREV_DAYS_DECISION_MAX,PREV_STATUS_APPROVED,PREV_STATUS_CANCELED,PREV_STATUS_REFUSED,PREV_STATUS_UNUSED_OFFER,PREV_APPROVAL_RATE,PREV_REFUSAL_RATE,PREV_CANCELLATION_RATE
0,100001,1,3951.000,3951.000,24835.50,24835.5,23787.00,23787.0,2520.0,24835.5,-1740.0,-1740,-1740,1,0,0,0,1.0,0.0,0.0
1,100002,1,9251.775,9251.775,179055.00,179055.0,179055.00,179055.0,0.0,179055.0,-606.0,-606,-606,1,0,0,0,1.0,0.0,0.0
2,100003,3,56553.990,98356.995,435436.50,900000.0,484191.00,1035882.0,3442.5,435436.5,-1305.0,-2341,-746,3,0,0,0,1.0,0.0,0.0
3,100004,1,5357.250,5357.250,24282.00,24282.0,20106.00,20106.0,4860.0,24282.0,-815.0,-815,-815,1,0,0,0,1.0,0.0,0.0
4,100005,2,4813.200,4813.200,22308.75,44617.5,20076.75,40153.5,4464.0,44617.5,-536.0,-757,-315,1,1,0,0,0.5,0.0,0.5


In [13]:
def aggregate_bureau(bureau):
    b = bureau.copy()

    num_agg = b.groupby("SK_ID_CURR").agg({
        "DAYS_CREDIT": ["count", "mean", "min", "max"],
        "CREDIT_DAY_OVERDUE": ["mean", "max"],
        "AMT_CREDIT_SUM": ["sum", "mean", "max"],
        "AMT_CREDIT_SUM_DEBT": ["sum", "mean"],
        "AMT_CREDIT_SUM_OVERDUE": ["sum", "max"],
        "CNT_CREDIT_PROLONG": ["sum"],
    })

    num_agg.columns = [
        "BURO_" + "_".join(c).upper()
        for c in num_agg.columns
    ]

    active = (
        b[b["CREDIT_ACTIVE"] == "Active"]
        .groupby("SK_ID_CURR")
        .size()
    )

    closed = (
        b[b["CREDIT_ACTIVE"] == "Closed"]
        .groupby("SK_ID_CURR")
        .size()
    )

    num_agg["BURO_ACTIVE_COUNT"] = active
    num_agg["BURO_CLOSED_COUNT"] = closed

    num_agg[
        ["BURO_ACTIVE_COUNT", "BURO_CLOSED_COUNT"]
    ] = num_agg[
        ["BURO_ACTIVE_COUNT", "BURO_CLOSED_COUNT"]
    ].fillna(0)

    return num_agg.reset_index()

def add_ratios(df):
    df = df.copy()

    df["CREDIT_INCOME_RATIO"] = (
        df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"]
    )

    df["ANNUITY_INCOME_RATIO"] = (
        df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"]
    )

    df["CREDIT_TERM"] = (
        df["AMT_ANNUITY"] / df["AMT_CREDIT"]
    )

    df["GOODS_CREDIT_RATIO"] = (
        df["AMT_GOODS_PRICE"] / df["AMT_CREDIT"]
    )

    df["INCOME_PER_PERSON"] = (
        df["AMT_INCOME_TOTAL"] / df["CNT_FAM_MEMBERS"]
    )

    return df


def prep_for_lgbm(df):
    df = df.copy()

    # Handle DAYS_EMPLOYED anomaly
    df["DAYS_EMPLOYED_ANOM"] = (
        df["DAYS_EMPLOYED"] == 365243
    ).astype("int8")

    df["DAYS_EMPLOYED"] = (
        df["DAYS_EMPLOYED"]
        .replace(365243, np.nan)
    )

    # Convert categorical columns to pandas category
    for col in df.select_dtypes(exclude="number").columns:
        df[col] = df[col].astype("category")

    # Reduce float64 memory usage
    for col in df.select_dtypes(include="float64").columns:
        df[col] = df[col].astype("float32")

    return df


print("prep_for_lgbm is ready.")

prep_for_lgbm is ready.


In [8]:
from sklearn.model_selection import StratifiedKFold

print("StratifiedKFold ready.")

StratifiedKFold ready.


In [9]:
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)

print("CV splitter ready.")

CV splitter ready.


In [10]:
app = pd.read_csv("../data/raw/application_train.csv")

bureau = pd.read_csv("../data/raw/bureau.csv")
buro_agg = aggregate_bureau(bureau)
del bureau

previous = pd.read_csv("../data/raw/previous_application.csv")
prev_agg = aggregate_previous(previous)
del previous

app_full = (
    app
    .merge(buro_agg, on="SK_ID_CURR", how="left")
    .merge(prev_agg, on="SK_ID_CURR", how="left")
)

print("Shape after both joins:", app_full.shape)

Shape after both joins: (307511, 157)


In [11]:
X_full = app_full.drop(columns=["TARGET", "SK_ID_CURR"])
y_full = app_full["TARGET"]

X_dev, X_test, y_dev, y_test = train_test_split(
    X_full,
    y_full,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_full
)

print("Development set:", X_dev.shape)
print("Test set:", X_test.shape)
print("Development default rate:", round(y_dev.mean(), 4))
print("Test default rate:", round(y_test.mean(), 4))

Development set: (246008, 155)
Test set: (61503, 155)
Development default rate: 0.0807
Test default rate: 0.0807


In [14]:
X_dev = prep_for_lgbm(add_ratios(X_dev))

X_dev["NO_BUREAU_HISTORY"] = (
    X_dev["BURO_DAYS_CREDIT_COUNT"].isna().astype("int8")
)

X_dev["NO_PREV_HISTORY"] = (
    X_dev["PREV_SK_ID_PREV_COUNT"].isna().astype("int8")
)

X_dev["PREV_CREDIT_APPLICATION_RATIO"] = (
    X_dev["PREV_AMT_CREDIT_MEAN"] /
    X_dev["PREV_AMT_APPLICATION_MEAN"]
).replace([np.inf, -np.inf], np.nan)

print("Shape:", X_dev.shape)
print(
    "Infinite values:",
    np.isinf(X_dev.select_dtypes("number")).sum().sum()
)

/tmp/ipykernel_9170/3108263113.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_dev["NO_BUREAU_HISTORY"] = (
/tmp/ipykernel_9170/3108263113.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_dev["NO_PREV_HISTORY"] = (
/tmp/ipykernel_9170/3108263113.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.c

Shape: (246008, 164)
Infinite values: 0


In [16]:
model = lgb.LGBMClassifier(
    random_state=RANDOM_STATE,
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    verbose=-1,
    n_jobs=2,
)

print("Model ready.")

Model ready.


In [17]:
aucs = cross_val_score(
    model,
    X_dev,
    y_dev,
    cv=cv,
    scoring="roc_auc",
    n_jobs=1
)

print(f"Full feature set: {aucs.mean():.4f} ± {aucs.std():.4f}")

Full feature set: 0.7686 ± 0.0011


In [18]:
pd.DataFrame([
    {"features": "application only",       "cv_auc": 0.7536, "cv_std": 0.0019},
    {"features": "+ ratios",               "cv_auc": 0.7601, "cv_std": 0.0009},
    {"features": "+ bureau",               "cv_auc": 0.7647, "cv_std": 0.0011},
    {"features": "+ previous application", "cv_auc": 0.7686, "cv_std": 0.0011},
]).to_csv("../reports/table_contributions.csv", index=False)